# 09. Support Vector Machine (SVM)

## Learning Objectives
- Understand the maximum margin principle of SVM
- Learn the role of support vectors
- Solve nonlinear problems with the kernel trick
- Tune hyperparameters C and gamma
- Solve regression problems with SVR

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.datasets import (
    make_blobs, make_classification, make_moons, make_circles,
    load_iris, load_breast_cancer, load_diabetes
)
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

# Font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 1. Linear SVM - Maximum Margin

The key idea of SVM is finding the optimal hyperplane that separates two classes.
It maximizes the margin to improve generalization performance.

In [ ]:
# Generate linearly separable data
X, y = make_blobs(n_samples=100, centers=2, random_state=6)

# Train linear SVM
clf = svm.SVC(kernel='linear', C=1000)
clf.fit(X, y)

print(f"Number of support vectors: {len(clf.support_vectors_)}")
print(f"Weights (w): {clf.coef_}")
print(f"Intercept (b): {clf.intercept_}")

In [ ]:
# Decision boundary and margin visualization
plt.figure(figsize=(10, 8))

# Data points
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=100, edgecolors='black')

# Decision boundary and margin
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Create grid
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
Z = clf.decision_function(xy).reshape(XX.shape)

# Draw decision boundary and margin
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           linestyles=['--', '-', '--'], linewidths=[1, 2, 1])

# Mark support vectors
ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
           s=200, linewidth=2, facecolors='none', edgecolors='green',
           label='Support Vectors')

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Linear SVM: Maximum Margin Classifier')
plt.legend()
plt.show()

## 2. Soft Margin - C Parameter

Real data is rarely perfectly linearly separable.
The C parameter balances misclassification and margin size.

- **Large C**: High penalty for misclassification -> narrow margin, overfitting risk
- **Small C**: Tolerates misclassification -> wide margin, better generalization

In [ ]:
# Noisy data
X, y = make_classification(
    n_samples=200, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    flip_y=0.1,  # 10% noise
    random_state=42
)

# Compare multiple C values
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
C_values = [0.1, 1, 100]

for ax, C in zip(axes, C_values):
    clf = svm.SVC(kernel='linear', C=C)
    clf.fit(X, y)

    # Decision boundary
    xlim = [X[:, 0].min() - 0.5, X[:, 0].max() + 0.5]
    ylim = [X[:, 1].min() - 0.5, X[:, 1].max() + 0.5]
    xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], 100),
                         np.linspace(ylim[0], ylim[1], 100))

    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.contour(xx, yy, Z, colors='k', levels=[-1, 0, 1],
               linestyles=['--', '-', '--'])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='black')
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=150, facecolors='none', edgecolors='green', linewidths=2)
    ax.set_title(f'C = {C}\nSupport Vectors: {len(clf.support_vectors_)}')

plt.tight_layout()
plt.show()

## 3. Kernel Trick - Nonlinear Classification

Kernel functions map data to higher-dimensional spaces to handle nonlinear patterns.

Key kernels:
- **linear**: K(x, y) = x.y
- **polynomial**: K(x, y) = (gamma*x.y + r)^d
- **rbf** (Gaussian): K(x, y) = exp(-gamma||x - y||^2)
- **sigmoid**: K(x, y) = tanh(gamma*x.y + r)

In [ ]:
# Generate nonlinear data
X_moons, y_moons = make_moons(n_samples=200, noise=0.1, random_state=42)
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=42)

# Kernel comparison
kernels = ['linear', 'poly', 'rbf']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for row, (X_data, y_data, name) in enumerate([(X_moons, y_moons, 'Moons'),
                                                (X_circles, y_circles, 'Circles')]):
    for col, kernel in enumerate(kernels):
        ax = axes[row, col]

        # Train SVM
        if kernel == 'poly':
            clf = svm.SVC(kernel=kernel, degree=3, gamma='scale')
        else:
            clf = svm.SVC(kernel=kernel, gamma='scale')
        clf.fit(X_data, y_data)

        # Decision boundary
        xlim = [X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5]
        ylim = [X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5]
        xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], 100),
                             np.linspace(ylim[0], ylim[1], 100))

        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)

        ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
        ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='coolwarm', edgecolors='black')
        ax.set_title(f'{name} - {kernel}\nAccuracy: {clf.score(X_data, y_data):.3f}')

plt.tight_layout()
plt.show()

## 4. RBF Kernel and gamma Parameter

In the RBF kernel, gamma determines the influence range of each data point.

- **Large gamma**: Narrow influence -> complex boundary, overfitting risk
- **Small gamma**: Wide influence -> simple boundary, underfitting risk

In [ ]:
# gamma effect visualization
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
gamma_values = [0.1, 1, 10, 100]

X, y = make_moons(n_samples=200, noise=0.1, random_state=42)

for ax, gamma in zip(axes, gamma_values):
    clf = svm.SVC(kernel='rbf', gamma=gamma, C=1)
    clf.fit(X, y)

    xlim = [X[:, 0].min() - 0.5, X[:, 0].max() + 0.5]
    ylim = [X[:, 1].min() - 0.5, X[:, 1].max() + 0.5]
    xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], 100),
                         np.linspace(ylim[0], ylim[1], 100))

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='black')
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=100, facecolors='none', edgecolors='green', linewidths=2)
    ax.set_title(f'gamma = {gamma}\nSVs: {len(clf.support_vectors_)}')

plt.tight_layout()
plt.show()

## 5. SVC - Real Data Classification

Performing multiclass classification with the Iris dataset.
**Important**: SVM is sensitive to feature scale, so scaling is essential.

In [ ]:
# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Why: SVM computes distances between data points, so features with larger scales
# dominate the margin calculation. fit_transform on train, transform on test
# prevents data leakage from test statistics.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Why: probability=True fits a Platt sigmoid calibration after SVM training,
# enabling predict_proba(). This adds computational cost but is needed for
# ROC curves, stacking, or soft voting ensembles.
svm_clf = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,  # Enable probability prediction
    random_state=42
)
svm_clf.fit(X_train_scaled, y_train)

# Prediction
y_pred = svm_clf.predict(X_test_scaled)

print("SVM Classification Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Number of support vectors: {len(svm_clf.support_vectors_)}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Probability prediction
y_proba = svm_clf.predict_proba(X_test_scaled[:5])

print("Probability predictions (first 5):")
print(f"Classes: {iris.target_names}")
print(y_proba)
print(f"\nPredicted classes: {y_pred[:5]}")
print(f"Actual classes: {y_test[:5]}")

## 6. Importance of Scaling

SVM is a distance-based algorithm, so differing feature scales degrade performance.

In [ ]:
# Scaling effect comparison
cancer = load_breast_cancer()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

# Without scaling
svm_no_scale = SVC(kernel='rbf', C=1, gamma='scale')
svm_no_scale.fit(X_train_c, y_train_c)
acc_no_scale = svm_no_scale.score(X_test_c, y_test_c)

# With scaling
scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

svm_scaled = SVC(kernel='rbf', C=1, gamma='scale')
svm_scaled.fit(X_train_c_scaled, y_train_c)
acc_scaled = svm_scaled.score(X_test_c_scaled, y_test_c)

print("Scaling effect:")
print(f"  Without scaling: {acc_no_scale:.4f}")
print(f"  With scaling:    {acc_scaled:.4f}")
print(f"  Improvement:     {(acc_scaled - acc_no_scale) * 100:.2f}%")

## 7. Hyperparameter Tuning - Grid Search

Simultaneously tune C and gamma to find the optimal combination.

In [ ]:
# Parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'kernel': ['rbf', 'poly']
}

# Grid Search
grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("\nGrid Search Results:")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Best CV score: {grid_search.best_score_:.4f}")
print(f"  Test score: {grid_search.score(X_test_scaled, y_test):.4f}")

In [ ]:
# C and gamma joint tuning visualization (RBF kernel only)
C_range = np.logspace(-2, 2, 5)
gamma_range = np.logspace(-3, 1, 5)

# Compute scores
scores = np.zeros((len(C_range), len(gamma_range)))

for i, C in enumerate(C_range):
    for j, gamma in enumerate(gamma_range):
        svm_clf = SVC(C=C, gamma=gamma, kernel='rbf')
        svm_clf.fit(X_train_c_scaled, y_train_c)
        scores[i, j] = svm_clf.score(X_test_c_scaled, y_test_c)

# Heatmap visualization
plt.figure(figsize=(10, 8))
plt.imshow(scores, interpolation='nearest', cmap='viridis')
plt.xlabel('gamma')
plt.ylabel('C')
plt.colorbar(label='Accuracy')
plt.xticks(np.arange(len(gamma_range)), [f'{g:.3f}' for g in gamma_range])
plt.yticks(np.arange(len(C_range)), [f'{c:.2f}' for c in C_range])
plt.title('SVM Hyperparameter Tuning (RBF Kernel)')

# Mark best point
best_i, best_j = np.unravel_index(scores.argmax(), scores.shape)
plt.scatter(best_j, best_i, marker='*', s=300, c='red', edgecolors='white')

plt.tight_layout()
plt.show()

print(f"Best C: {C_range[best_i]:.2f}")
print(f"Best gamma: {gamma_range[best_j]:.3f}")
print(f"Best accuracy: {scores.max():.4f}")

## 8. SVR - Support Vector Regression

Applying SVM to regression problems.
Errors within the epsilon-tube are ignored; only errors outside the tube are penalized.

In [ ]:
# Load data
diabetes = load_diabetes()
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_d_scaled = scaler.fit_transform(X_train_d)
X_test_d_scaled = scaler.transform(X_test_d)

# Why: epsilon defines a "tube" around the prediction — errors within this tube
# incur zero loss. This makes SVR robust to small noise. C=100 (high) prioritizes
# fitting the data tightly over having a wide margin.
svr = SVR(
    kernel='rbf',
    C=100,
    epsilon=0.1,  # Tube width: errors within are ignored
    gamma='scale'
)
svr.fit(X_train_d_scaled, y_train_d)

# Prediction
y_pred_d = svr.predict(X_test_d_scaled)

print("SVR Regression Results:")
print(f"  MSE: {mean_squared_error(y_test_d, y_pred_d):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_d, y_pred_d)):.4f}")
print(f"  R2: {r2_score(y_test_d, y_pred_d):.4f}")
print(f"  Number of support vectors: {len(svr.support_vectors_)}")

In [ ]:
# Visualization
plt.figure(figsize=(8, 6))
plt.scatter(y_test_d, y_pred_d, alpha=0.7, edgecolors='black')
plt.plot([y_test_d.min(), y_test_d.max()], [y_test_d.min(), y_test_d.max()], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'SVR Regression (R² = {r2_score(y_test_d, y_pred_d):.4f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Multiclass Classification Strategy

SVM is a binary classifier, so multiclass is handled via the following strategies:

- **OvO (One-vs-One)**: k(k-1)/2 classifiers, SVC default
- **OvR (One-vs-Rest)**: k classifiers, LinearSVC default

In [ ]:
# OvO (default)
svm_ovo = SVC(kernel='rbf', decision_function_shape='ovo')
svm_ovo.fit(X_train_scaled, y_train)
print(f"OvO accuracy: {svm_ovo.score(X_test_scaled, y_test):.4f}")

# OvR
svm_ovr = SVC(kernel='rbf', decision_function_shape='ovr')
svm_ovr.fit(X_train_scaled, y_train)
print(f"OvR accuracy: {svm_ovr.score(X_test_scaled, y_test):.4f}")

# LinearSVC (OvR default)
linear_svc = LinearSVC(dual=True, max_iter=10000)
linear_svc.fit(X_train_scaled, y_train)
print(f"LinearSVC accuracy: {linear_svc.score(X_test_scaled, y_test):.4f}")

## 10. Kernel Comparison - Practical

Comparing kernel performance on breast cancer data.

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

print("Performance comparison by kernel (Breast Cancer):")
print("-" * 50)

for kernel in kernels:
    if kernel == 'poly':
        svm_model = SVC(kernel=kernel, degree=3, gamma='scale')
    else:
        svm_model = SVC(kernel=kernel, gamma='scale')

    svm_model.fit(X_train_c_scaled, y_train_c)
    acc = svm_model.score(X_test_c_scaled, y_test_c)
    print(f"  {kernel:8s}: {acc:.4f} (SVs: {len(svm_model.support_vectors_)})")

## Summary

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Support Vectors** | Key data points located at the margin boundary |
| **Margin** | Distance between the decision boundary and support vectors |
| **C** | Regularization parameter (large: narrow margin, small: wide margin) |
| **gamma** | RBF kernel range (large: narrow influence, small: wide influence) |
| **Kernel** | Function that maps data to higher dimensions |

### SVM Usage Checklist

1. **Scaling is essential**: Apply StandardScaler or MinMaxScaler
2. **Kernel selection**: Linearly separable -> linear, nonlinear -> rbf
3. **Parameter tuning**: Tune C and gamma with GridSearchCV
4. **Large data**: Use LinearSVC or SGDClassifier
5. **Probabilities needed**: Set probability=True (additional cost)

### Pros and Cons

**Pros**:
- Effective for high-dimensional data
- Memory efficient (only stores support vectors)
- Solves nonlinear problems with various kernels

**Cons**:
- Slow on large data (O(n^2) ~ O(n^3))
- Scaling required
- Parameter tuning needed

### Next Steps
- k-Nearest Neighbors (kNN)
- Naive Bayes
- Ensemble methods